# 法說會音檔 → 逐字稿（Colab）

1. 從公開資訊觀測站 [t100sb02_1](https://mopsov.twse.com.tw/mops/web/t100sb02_1) 抓取指定月份（預設當月）的法說會清單
2. 下載「影音連結資訊」中的影音（支援 irconference mp4、webpro 串流、YouTube，及 yt-dlp 能解析的網站）
3. 在 GPU 上用 faster-whisper 轉成繁體中文逐字稿；轉錄時背景會預先下載後面幾場（`DOWNLOAD_WORKERS`）
4. 存到 Google 雲端硬碟；`manifest.json` 記錄已處理的場次，重跑時自動跳過
   - 失敗分為「永久」（該連結不是影音、影片已刪除）與「暫時」（YouTube 驗證、網路問題），暫時性失敗隔一段時間自動重試
   - 公司之後補上新的影音連結時，會自動重新處理
   - 要重做特定場次，把代號_日期加入設定中的 `REDO_KEYS`

> 也可以不用 Colab，改在本機執行 `npm run transcribe:earnings`（用 agy 雲端轉錄）。兩邊共用這份 `manifest.json`，對方處理過的場次會自動跳過；
> 兩者的逐字稿都會由 `npm run summarize:earnings` 摘要。

**請先切換執行階段：執行階段 → 變更執行階段類型 → GPU（T4 / L4 / A100），執行階段版本選 `2026.07`**

> 較新的執行階段改用 CUDA 13，faster-whisper（CTranslate2）需要 CUDA 12，轉錄時會出現 `Library libcublas.so.12 is not found or cannot be loaded`。

In [ ]:
!pip install -q faster-whisper yt-dlp beautifulsoup4 opencc-python-reimplemented
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import datetime as dt
from pathlib import Path
from zoneinfo import ZoneInfo

TZ = ZoneInfo('Asia/Taipei')
TODAY = dt.datetime.now(TZ).date()

# ===== 設定 =====
YEAR, MONTH = TODAY.year, TODAY.month      # 要抓的月份（西元），預設當月；補抓上個月可改成例如 2026, 8
MARKETS = ['sii', 'otc']                   # sii 上市、otc 上櫃、rotc 興櫃、pub 公開發行
ONLY_CODES = []                            # 只處理特定公司，例如 ['2330', '2317']；空 = 全部
REDO_KEYS = []                             # 強制重做的場次（代號_YYYYMMDD），例如 ['1232_20260911']；重做完請清空
URL_OVERRIDES = {}                         # 手動指定影音網址（觀測站連結錯誤時），例如 {'1217_20260911': 'https://youtu.be/xxxx'}；
                                           # 會優先於觀測站連結嘗試，可填字串或清單
MAX_ITEMS = None                           # 本次最多處理幾場（測試用），None = 不限
DOWNLOAD_WORKERS = 2                       # GPU 轉錄時，背景同時預先下載幾場；YouTube 常要求驗證時可調為 1
MAX_ATTEMPTS = 5                           # 暫時性失敗（YouTube 驗證、網路逾時…）最多嘗試幾次；出現新連結時重新計算
RETRY_AFTER_HOURS = 24                     # 暫時性失敗後，隔多久才再試
MIN_TRANSCRIPT_CHARS = 500                 # 逐字稿內文少於此字數視為失敗（靜音、片頭音樂或抓錯影片）
MANIFEST_KEEP_MONTHS = 6                   # manifest 保留最近幾個月的紀錄，更舊的自動清除
WHISPER_MODEL = 'large-v3-turbo'                 # 記憶體不夠可改 'medium'
YTDLP_COOKIES = None                       # YouTube 要求登入驗證時，上傳 cookies.txt 並填路徑
YT_PLAYER_CLIENTS = 'tv,web_safari,mweb'   # YouTube 播放器用戶端，依序嘗試；None = yt-dlp 預設
YT_SLEEP_SECONDS = 10                      # 每支 YouTube 下載前等待秒數，降低被判定為機器人；0 = 不等

DRIVE_ROOT = Path('/content/drive/MyDrive/投資報告/EarningsCall')
OUT_DIR = DRIVE_ROOT / f'{YEAR}{MONTH:02d}'
MANIFEST_PATH = DRIVE_ROOT / 'manifest.json'
WORK = Path('/content/work')

for d in (OUT_DIR, WORK):
    d.mkdir(parents=True, exist_ok=True)
print('輸出目錄：', OUT_DIR)

## 共用函式（manifest 與重試判斷）

In [ ]:
import json
import os
import re

# 這些錯誤代表「這條連結」再試也不會成功（公司官網頁面、影片已刪除、內容不是法說會…），之後不再嘗試該連結；
# 其他錯誤（YouTube 機器人驗證、網路逾時、限流…）視為暫時性，隔 RETRY_AFTER_HOURS 再試。
# 證交所自己的連結（irconference / webpro）常是先公告、影片稍後才上架，一律視為暫時性。
PERMANENT_ERROR_PATTERNS = [
    'Unsupported URL', 'yt-dlp 沒有產生檔案', '逐字稿過短', 'HTTP Error 404', 'HTTP Error 410',
    'Video unavailable', 'Private video', 'This video has been removed', 'is not a valid URL',
]


def load_manifest():
    if MANIFEST_PATH.exists():
        return json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
    return {}


def record_rank(rec):
    """兩邊都有同一場紀錄時的取捨：已完成優先，其次取最後更新時間較新者。"""
    return rec.get('status') == 'done', rec.get('updated') or rec.get('last_attempt') or ''


def save_manifest(manifest):
    # 本地 npm run transcribe:earnings 也會讀寫同一份 manifest：先合併雲端硬碟上的最新內容，避免蓋掉對方的紀錄
    if MANIFEST_PATH.exists():
        for key, rec in json.loads(MANIFEST_PATH.read_text(encoding='utf-8')).items():
            if key not in manifest or record_rank(rec) > record_rank(manifest[key]):
                manifest[key] = rec
        prune_manifest(manifest)  # 已清除的過期紀錄不要被合併回來
    tmp = MANIFEST_PATH.with_suffix('.tmp')
    tmp.write_text(json.dumps(manifest, ensure_ascii=False, indent=1), encoding='utf-8')
    os.replace(tmp, MANIFEST_PATH)


def safe_name(s):
    return re.sub(r'[\\/:*?"<>|\s]+', '_', s).strip('_')


def apply_url_overrides(key, urls):
    """手動指定的網址排在最前面，觀測站原本的連結留作備援。"""
    extra = URL_OVERRIDES.get(key, [])
    extra = [extra] if isinstance(extra, str) else list(extra)
    return list(dict.fromkeys(extra + urls))


def short_error(message):
    """從 yt-dlp / ffmpeg 的輸出挑出關鍵的一行（優先取最後一個 ERROR 行）。"""
    lines = [l.strip() for l in message.splitlines() if l.strip()]
    errs = [l for l in lines if l.startswith('ERROR')]
    return (errs or lines or [message])[-1]


def is_permanent_error(url, message):
    if 'twse.com.tw' in url:
        return False
    return any(p in message for p in PERMANENT_ERROR_PATTERNS)


def live_urls(rec, urls):
    """排除已確認無法使用的連結。"""
    dead = set(rec.get('dead_urls', []))
    return [u for u in urls if u not in dead]


def has_new_urls(rec, urls):
    """清單中出現尚未嘗試過的連結（例如公司後來補上 YouTube）。"""
    tried = set(rec.get('tried_urls', []))
    return any(u not in tried for u in live_urls(rec, urls))


def retry_blocker(rec, urls):
    """回傳這場暫不處理的原因；None 代表可以處理。"""
    if not live_urls(rec, urls):
        return '所有連結皆無法使用'
    if has_new_urls(rec, urls):
        return None
    if rec.get('attempts', 0) >= MAX_ATTEMPTS:
        return '已達重試上限'
    last = rec.get('last_attempt')
    if last:
        next_try = dt.datetime.fromisoformat(last) + dt.timedelta(hours=RETRY_AFTER_HOURS)
        if dt.datetime.now(TZ) < next_try:
            return f'{next_try:%m/%d %H:%M} 後重試'
    return None


def prune_manifest(manifest):
    """清除 MANIFEST_KEEP_MONTHS 個月以前的紀錄（補抓舊月份時保留該月以後的紀錄）。"""
    y, m = divmod(TODAY.year * 12 + TODAY.month - 1 - MANIFEST_KEEP_MONTHS, 12)
    cutoff = min(dt.date(y, m + 1, 1), dt.date(YEAR, MONTH, 1)).isoformat()
    old = [k for k, v in manifest.items() if v.get('date', cutoff) < cutoff]
    for k in old:
        del manifest[k]
    return len(old)

## 抓取當月法說會清單

In [ ]:
import re
import time

import pandas as pd
import requests
from bs4 import BeautifulSoup

BASE = 'https://mopsov.twse.com.tw/mops/web'
UA = ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
      '(KHTML, like Gecko) Chrome/140.0 Safari/537.36')
S = requests.Session()
S.headers.update({'User-Agent': UA, 'Referer': f'{BASE}/t100sb02_1'})

DATE_RE = re.compile(r'(\d{2,3})/(\d{1,2})/(\d{1,2})')
URL_RE = re.compile(r'https?://[^\s<>"\']+')


def url_priority(url):
    """中文場 mp4 > 其他 irconference > webpro 串流 > YouTube > 其他網站。"""
    if 'irconference.twse.com.tw' in url:
        return 0 if '_ch' in url else 1
    if 'webpro.twse.com.tw' in url:
        return 2
    if 'youtu' in url:
        return 3
    return 4


def fetch_market(typek, year, month):
    data = {'encodeURIComponent': 1, 'step': 1, 'firstin': 'true', 'off': 1,
            'TYPEK': typek, 'year': year - 1911, 'month': f'{month:02d}', 'co_id': ''}
    for attempt in range(4):
        try:
            r = S.post(f'{BASE}/ajax_t100sb02_1', data=data, timeout=60)
            r.encoding = 'utf-8'
            if '查無資料' in r.text:
                return []
            if 'myTable' in r.text:
                break
        except requests.RequestException as e:
            print(f'  {typek} 第 {attempt + 1} 次失敗：{e}')
        time.sleep(5 * (attempt + 1))
    else:
        raise RuntimeError(f'{typek} 清單抓取失敗（可能被限流，稍後再試）')

    soup = BeautifulSoup(r.text, 'html.parser')
    rows = []
    for tr in soup.select('#myTable tr[data-type=body]'):
        tds = tr.find_all('td')
        if len(tds) < 10:
            continue
        dates = DATE_RE.findall(tds[2].get_text())
        if not dates:
            continue
        y, mo, d = dates[-1]  # 日期區間（例如 115/08/17 至 115/09/15）取結束日
        media = tds[9]
        urls = [a['href'].strip() for a in media.find_all('a', href=True)]
        urls += URL_RE.findall(media.get_text(' '))
        urls = sorted(dict.fromkeys(u for u in urls if u.startswith('http')), key=url_priority)
        rows.append({
            'market': typek,
            'code': tds[0].get_text(strip=True),
            'name': tds[1].get_text(strip=True),
            'date': dt.date(int(y) + 1911, int(mo), int(d)),
            'date_text': tds[2].get_text(' ', strip=True),
            'time': tds[3].get_text(strip=True),
            'summary': tds[5].get_text(' ', strip=True),
            'urls': urls,
        })
    return rows


all_rows = []
for mk in MARKETS:
    rows = fetch_market(mk, YEAR, MONTH)
    print(f'{mk}: {len(rows)} 場')
    all_rows += rows
    time.sleep(3)

df = pd.DataFrame(all_rows, columns=['market', 'code', 'name', 'date', 'date_text', 'time', 'summary', 'urls'])
if ONLY_CODES:
    df = df[df.code.isin(ONLY_CODES)]
df['key'] = df.code + '_' + df.date.map(lambda d: d.strftime('%Y%m%d'))
df = df.drop_duplicates('key')
df['urls'] = [apply_url_overrides(k, u) for k, u in zip(df.key, df.urls)]
missing = set(URL_OVERRIDES) - set(df.key)
if missing:
    print(f'⚠️ URL_OVERRIDES 中這些場次不在本月清單：{sorted(missing)}')
print(f'共 {len(df)} 場，已召開 {int((df.date <= TODAY).sum())} 場，'
      f'已召開且有影音連結 {int(((df.date <= TODAY) & (df.urls.str.len() > 0)).sum())} 場')
df.head()

## 已處理紀錄（跳過已爬取的場次）

In [ ]:
manifest = load_manifest()
pruned = prune_manifest(manifest)

todo = []
counts = {'已完成': 0, '補登已完成': 0, '尚未召開或無影音': 0, '等待重試': 0, '放棄': 0}
for row in df.itertuples():
    stem = f'{row.date:%Y%m%d}_{row.code}_{safe_name(row.name)}'
    txt = OUT_DIR / f'{stem}.txt'
    if row.key in REDO_KEYS:
        manifest.pop(row.key, None)  # 強制重做：清掉舊紀錄，成功後覆蓋舊逐字稿
    elif manifest.get(row.key, {}).get('status') == 'done':
        counts['已完成'] += 1
        continue
    elif txt.exists() or txt.with_suffix('.md').exists():
        # 逐字稿或摘要已在但 manifest 沒記到（例如上次存檔前中斷，或本地 transcribe:earnings 已處理並上傳摘要），補登紀錄
        manifest[row.key] = {'code': row.code, 'name': row.name, 'date': str(row.date),
                             'status': 'done', 'file': str(txt)}
        counts['補登已完成'] += 1
        continue

    if row.date > TODAY or not row.urls:
        counts['尚未召開或無影音'] += 1
        continue  # 下次再試
    blocker = retry_blocker(manifest.get(row.key, {}), row.urls)
    if blocker:
        counts['等待重試' if blocker.endswith('後重試') else '放棄'] += 1
        continue
    todo.append((row, stem))

if pruned or counts['補登已完成']:
    save_manifest(manifest)
if MAX_ITEMS:
    todo = todo[:MAX_ITEMS]
print(f'manifest 清除 {pruned} 筆過期紀錄；' + '、'.join(f'{k} {v} 場' for k, v in counts.items()))
print(f'本次待處理 {len(todo)} 場')
for row, stem in todo[:20]:
    print(' ', stem, row.urls[0])

## 下載音檔

In [ ]:
import subprocess

GDRIVE_ID_RE = re.compile(r'drive\.google\.com/(?:file/d/|open\?id=|uc\?(?:[^#]*&)?id=)([\w-]+)')


def resolve_url(url):
    """把網頁連結轉成 yt-dlp 可直接下載的媒體網址。"""
    if url.startswith('http://irconference.twse.com.tw'):
        return 'https://' + url[len('http://'):]
    if 'webpro.twse.com.tw' in url:
        html = S.get(url, timeout=60).text
        m = re.search(r'https://webprovod\.twse\.com\.tw/[^"\\\s]+?playlist\.m3u8', html)
        if m:
            return m.group(0)
    return url

def run(cmd, timeout):
    p = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    if p.returncode != 0:
        raise RuntimeError((p.stderr or p.stdout).strip()[-500:])


def download_gdrive(file_id, dest):
    """直接下載 Google 雲端硬碟原始檔。
    經 yt-dlp 下載雲端硬碟上的 mp3 會出現逐字稿只剩 0～300 字的情況，改為直接下載原始檔。"""
    r = requests.get('https://drive.usercontent.google.com/download',
                     params={'id': file_id, 'export': 'download', 'confirm': 't'},
                     headers={'User-Agent': UA}, stream=True, timeout=60)
    r.raise_for_status()
    if 'text/html' in r.headers.get('Content-Type', ''):
        raise RuntimeError('Google Drive 回傳網頁而非檔案（可能未設為「知道連結的任何人」或超過下載配額）')
    with open(dest, 'wb') as f:
        for chunk in r.iter_content(1 << 20):
            f.write(chunk)


def download_audio(url, stem):
    """下載影音並轉成 16kHz 單聲道 mp3，回傳本機路徑。"""
    for f in WORK.glob(f'{stem}*'):
        f.unlink()
    gdrive = GDRIVE_ID_RE.search(url)
    if gdrive:
        download_gdrive(gdrive.group(1), WORK / f'{stem}_raw.bin')
    else:
        cmd = ['yt-dlp', '--no-playlist', '--no-progress', '-f', 'bestaudio/best',
               '--user-agent', UA, '-o', str(WORK / f'{stem}_raw.%(ext)s')]
        if YTDLP_COOKIES:
            cmd += ['--cookies', YTDLP_COOKIES]
        if 'youtu' in url:
            if YT_PLAYER_CLIENTS:
                cmd += ['--extractor-args', f'youtube:player_client={YT_PLAYER_CLIENTS}']
            if YT_SLEEP_SECONDS:
                cmd += ['--sleep-requests', '2', '--sleep-interval', str(YT_SLEEP_SECONDS)]
        run(cmd + [resolve_url(url)], timeout=3600)

    raw = next((p for p in WORK.glob(f'{stem}_raw.*') if p.suffix != '.part'), None)
    if raw is None:
        raise RuntimeError('yt-dlp 沒有產生檔案')
    out = WORK / f'{stem}.mp3'
    run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(raw),
         '-vn', '-ac', '1', '-ar', '16000', '-b:a', '48k', str(out)], timeout=1800)
    raw.unlink()
    return out

## 載入 Whisper 模型

In [ ]:
import subprocess

import numpy as np
import torch
from faster_whisper import WhisperModel
from opencc import OpenCC

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = WhisperModel(WHISPER_MODEL, device=device,
                     compute_type='float16' if device == 'cuda' else 'int8')
cc = OpenCC('s2twp')  # 保證輸出為台灣繁體
print('Whisper on', device)


def fmt_ts(sec):
    h, rem = divmod(int(sec), 3600)
    m, s = divmod(rem, 60)
    return f'{h:02d}:{m:02d}:{s:02d}'


def load_audio(path, sr=16000):
    """用 ffmpeg 解碼成 float32 陣列。
    faster-whisper 內建用 PyAV 解碼，新版 av 移除了 metadata_errors 參數會直接報錯，因此改走 ffmpeg。"""
    p = subprocess.run(['ffmpeg', '-nostdin', '-loglevel', 'error', '-i', str(path),
                        '-f', 's16le', '-ac', '1', '-ar', str(sr), '-'],
                       capture_output=True, check=True)
    return np.frombuffer(p.stdout, np.int16).astype(np.float32) / 32768.0


def run_whisper(audio, row, lang, strict):
    """strict=False 時關閉 VAD 與靜音判斷：有些錄音（現場底噪、會議系統壓縮）是清楚的人聲，
    卻被 VAD 或 no_speech 判斷整段濾掉，只剩幾百字。"""
    opts = {} if strict else {'no_speech_threshold': None, 'log_prob_threshold': None}
    segments, info = model.transcribe(
        audio,
        language=lang,
        beam_size=5,
        vad_filter=strict,
        condition_on_previous_text=False,  # 降低長音檔的重複幻覺
        initial_prompt=f'以下是{row.name}法人說明會的逐字稿，內容包含營收、毛利率、展望等財務用語。',
        **opts,
    )
    lines = []
    for seg in segments:
        text = seg.text.strip()
        if text:
            lines.append(f'[{fmt_ts(seg.start)}] {cc.convert(text) if lang == "zh" else text}')
    return lines, info


def transcribe(audio_path, row, url):
    lang = 'en' if re.search(r'_en\b', url) else 'zh'
    audio = load_audio(audio_path)
    lines, info = run_whisper(audio, row, lang, strict=True)
    if sum(map(len, lines)) < MIN_TRANSCRIPT_CHARS:
        kept = getattr(info, 'duration_after_vad', None)
        print(f'   ⚠️ 只辨識出 {sum(map(len, lines))} 字（VAD 保留 {fmt_ts(kept) if kept is not None else "?"}'
              f' / 全長 {fmt_ts(info.duration)}），關閉 VAD 與靜音判斷重跑')
        lines, info = run_whisper(audio, row, lang, strict=False)
    header = [
        f'公司：{row.code} {row.name}',
        f'日期：{row.date_text} {row.time}',
        f'摘要：{row.summary}',
        f'來源：{url}',
        f'長度：{fmt_ts(info.duration)}',
        '',
    ]
    return '\n'.join(header + lines) + '\n'

## 主流程（可中斷，重跑會從未完成的場次繼續）

In [ ]:
from concurrent.futures import ThreadPoolExecutor

from tqdm.auto import tqdm


def first_live_url(row):
    """與下方主迴圈相同的連結判斷 (手動指定的網址不算失效)，回傳第一個會嘗試的連結。"""
    manual = set(apply_url_overrides(row.key, []))
    dead = set(manifest.get(row.key, {}).get('dead_urls', [])) - manual
    return next((u for u in row.urls if u not in dead), None)


# GPU 一次只轉錄一場；下載走背景執行緒，預先抓後面 DOWNLOAD_WORKERS 場，轉錄時不必等下載。
# 只預先下載每場的第一個連結，失敗改用備援連結時仍在主迴圈當場下載。
pool = ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS)
prefetched = {}  # todo 索引 → (url, future)


def prefetch(i):
    if i < len(todo) and i not in prefetched:
        row, stem = todo[i]
        url = first_live_url(row)
        if url:
            prefetched[i] = (url, pool.submit(download_audio, url, stem))


stats = {'done': 0, 'failed': 0}
try:
    for i, (row, stem) in enumerate(tqdm(todo)):
        for j in range(i, i + DOWNLOAD_WORKERS + 1):
            prefetch(j)
        ahead_url, ahead = prefetched.pop(i, (None, None))

        rec = manifest.setdefault(row.key, {'code': row.code, 'name': row.name, 'date': str(row.date)})
        manual = set(apply_url_overrides(row.key, []))
        rec['dead_urls'] = [u for u in rec.get('dead_urls', []) if u not in manual]
        if has_new_urls(rec, row.urls):
            rec['attempts'] = 0  # 出現新連結，重新計算嘗試次數
        rec['attempts'] = rec.get('attempts', 0) + 1
        rec['last_attempt'] = dt.datetime.now(TZ).isoformat(timespec='seconds')
        rec['urls'] = row.urls
        errors = []
        for url in live_urls(rec, row.urls):
            try:
                t0 = time.time()
                audio = ahead.result() if url == ahead_url else download_audio(url, stem)
                text = transcribe(audio, row, url)
                body_chars = len(text.partition('\n\n')[2])
                if body_chars < MIN_TRANSCRIPT_CHARS:
                    audio.unlink()
                    length = re.search(r'長度：(\S+)', text).group(1)
                    raise RuntimeError(f'逐字稿過短（{body_chars} 字，音檔長度 {length}），'
                                       '可能是靜音、片頭音樂、抓錯影片或下載不完整')
                (OUT_DIR / f'{stem}.txt').write_text(text, encoding='utf-8')
                audio.unlink()
                for k in ('error', 'tried_urls', 'dead_urls', 'urls'):
                    rec.pop(k, None)
                rec.update(status='done', url=url, file=str(OUT_DIR / f'{stem}.txt'),
                           updated=dt.datetime.now(TZ).isoformat(timespec='seconds'))
                stats['done'] += 1
                print(f'✅ {stem}（{time.time() - t0:.0f}s）')
                break
            except Exception as e:
                message = str(e)
                rec['tried_urls'] = list(dict.fromkeys(rec.get('tried_urls', []) + [url]))
                # URL_OVERRIDES 是手動指定的，不標成失效，改程式後重跑就會再試
                permanent = url not in manual and is_permanent_error(url, message)
                if permanent:
                    rec['dead_urls'] = list(dict.fromkeys(rec.get('dead_urls', []) + [url]))
                # 每個連結各記一行關鍵錯誤，避免覆寫網址的錯誤被後面備援連結的訊息蓋掉
                errors.append(f'[{"永久" if permanent else "暫時"}] {url} → {short_error(message)[:300]}')
        else:
            rec.update(status='failed', error='\n'.join(errors) or '無可用連結')
            stats['failed'] += 1
            print(f'❌ {stem}')
            for err in errors or ['無可用連結']:
                print(f'   {err[:250]}')
        save_manifest(manifest)
finally:
    # 中斷時取消尚未開始的下載 (已開始的會跑完，殘留音檔下次下載同場時會先清掉)
    pool.shutdown(wait=False, cancel_futures=True)

print(stats)

## 失敗清單

In [ ]:
import pandas as pd

# 直接讀雲端的 manifest，只需先執行「設定」與「共用函式」兩格，不必抓清單或載入 Whisper
failed = {k: v for k, v in load_manifest().items() if v.get('status') == 'failed'}


def next_action(key, rec):
    """用與主流程相同的連結清單（含 URL_OVERRIDES）判斷下次執行會不會重試。"""
    if 'urls' not in rec and key not in URL_OVERRIDES:
        return '下次執行重新判斷'  # 舊版紀錄沒有 urls
    return retry_blocker(rec, apply_url_overrides(key, rec.get('urls', []))) or '下次執行會重試'


if failed:
    table = pd.DataFrame.from_dict(failed, orient='index')
    table['last_attempt'] = [v.get('last_attempt', '') for v in failed.values()]
    table['next'] = [next_action(k, v) for k, v in failed.items()]
    with pd.option_context('display.max_colwidth', None, 'display.max_rows', None, 'display.width', None):
        display(table[['code', 'name', 'date', 'attempts', 'last_attempt', 'next', 'error']]
                .style.set_properties(subset=['error'], **{'text-align': 'left', 'white-space': 'pre-wrap', 'min-width': '600px'}))
else:
    print('沒有失敗的場次')